# J3 - Serveur MCP d'outils d'analyse symbolique

Ce notebook realise le projet **J3 - Serveur MCP d'outils d'analyse symbolique** sous une forme executable et commentee.

Il contient :

- un serveur JSON-RPC inspire du Model Context Protocol, avec les methodes `initialize`, `tools/list` et `tools/call` ;
- trois outils symboliques exposes derriere une interface unique : solveur SAT/SMT, verificateur de preuves, raisonneur OWL-lite ;
- une gestion de sessions et de contexte ;
- des exemples d'utilisation pour chaque outil ;
- une evaluation simple de precision et de robustesse sur des benchmarks de raisonnement.

Le choix technique est volontairement pedagogique : tout fonctionne en Python pur, sans dependance obligatoire. Pour un serveur de production, on remplacerait le mini-SMT borne par Z3, le raisonneur OWL-lite par Owlready2/RDFLib ou un moteur OWL-RL, et la boucle locale par un transport MCP stdio ou HTTP complet.

## Architecture

Le serveur suit le motif suivant :

1. Le client envoie une requete JSON-RPC.
2. Le serveur valide la methode et la session.
3. `tools/call` route l'appel vers l'outil symbolique demande.
4. L'outil renvoie un resultat structure, plus un texte exploitable par un LLM.
5. Le serveur memorise l'appel et son resultat dans le contexte de session.

Les noms d'outils exposes sont :

- `symbolic.sat_smt.solve` : SAT en CNF par DPLL, ou mini-SMT borne sur entiers.
- `symbolic.proof.verify` : verification de preuves par resolution propositionnelle.
- `symbolic.owl.reason` : saturation OWL-lite/RDFS avec detection d'incoherences simples.

In [1]:
from __future__ import annotations

import ast
import itertools
import json
import operator
import re
import sys
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, FrozenSet, Iterable, List, Optional, Sequence, Set, Tuple

JSON = Dict[str, Any]
Clause = List[int]
LiteralClause = FrozenSet[str]
Triple = Tuple[str, str, str]


def pretty(obj: Any) -> None:
    """Affichage JSON lisible dans le notebook."""
    print(json.dumps(obj, ensure_ascii=False, indent=2, sort_keys=False))

## Outil 1 - Solveur SAT/SMT

L'outil accepte deux familles d'entrees :

- SAT : une formule CNF sous forme de clauses d'entiers, par exemple `[[1, -2], [2]]`.
- SMT borne : une liste de contraintes arithmetiques sur des entiers, par exemple `x + y == 3`, avec des bornes finies.

Le solveur SAT utilise DPLL avec propagation unitaire et litteraux purs. Le solveur SMT est un enumerateur borne et securise : il parse les expressions avec `ast` et n'autorise ni appels de fonctions ni acces systeme.

In [2]:
def normalize_int_clauses(raw_clauses: Sequence[Sequence[int]]) -> List[Clause]:
    clauses: List[Clause] = []
    for raw_clause in raw_clauses:
        clause: List[int] = []
        seen: Set[int] = set()
        tautological = False
        for lit in raw_clause:
            lit = int(lit)
            if lit == 0:
                continue
            if -lit in seen:
                tautological = True
                break
            if lit not in seen:
                seen.add(lit)
                clause.append(lit)
        if not tautological:
            clauses.append(clause)
    return clauses


def eval_literal(lit: int, assignment: Dict[int, bool]) -> Optional[bool]:
    var = abs(lit)
    if var not in assignment:
        return None
    return assignment[var] if lit > 0 else not assignment[var]


def simplify_formula(clauses: List[Clause], assignment: Dict[int, bool]) -> Optional[List[Clause]]:
    simplified: List[Clause] = []
    for clause in clauses:
        pending: Clause = []
        satisfied = False
        for lit in clause:
            value = eval_literal(lit, assignment)
            if value is True:
                satisfied = True
                break
            if value is None:
                pending.append(lit)
        if satisfied:
            continue
        if not pending:
            return None
        simplified.append(pending)
    return simplified


def find_unit_literal(clauses: List[Clause]) -> Optional[int]:
    for clause in clauses:
        if len(clause) == 1:
            return clause[0]
    return None


def find_pure_literal(clauses: List[Clause], assignment: Dict[int, bool]) -> Optional[int]:
    signs: Dict[int, Set[bool]] = {}
    for clause in clauses:
        for lit in clause:
            var = abs(lit)
            if var in assignment:
                continue
            signs.setdefault(var, set()).add(lit > 0)
    for var, var_signs in sorted(signs.items()):
        if len(var_signs) == 1:
            return var if True in var_signs else -var
    return None


def dpll(clauses: List[Clause], assignment: Optional[Dict[int, bool]] = None) -> Optional[Dict[int, bool]]:
    assignment = dict(assignment or {})
    while True:
        simplified = simplify_formula(clauses, assignment)
        if simplified is None:
            return None
        if not simplified:
            return assignment

        unit = find_unit_literal(simplified)
        if unit is not None:
            assignment[abs(unit)] = unit > 0
            continue

        pure = find_pure_literal(simplified, assignment)
        if pure is not None:
            assignment[abs(pure)] = pure > 0
            continue

        variables = sorted({abs(lit) for clause in simplified for lit in clause if abs(lit) not in assignment})
        if not variables:
            return assignment

        chosen = variables[0]
        for value in (True, False):
            branch = dict(assignment)
            branch[chosen] = value
            result = dpll(clauses, branch)
            if result is not None:
                return result
        return None


def symbol_name(var: int, symbols: Optional[Any]) -> str:
    if isinstance(symbols, list) and 1 <= var <= len(symbols):
        return str(symbols[var - 1])
    if isinstance(symbols, dict):
        if str(var) in symbols:
            return str(symbols[str(var)])
        if var in symbols:
            return str(symbols[var])
    return f"x{var}"


def solve_sat(clauses: Sequence[Sequence[int]], symbols: Optional[Any] = None) -> JSON:
    normalized = normalize_int_clauses(clauses)
    variables = sorted({abs(lit) for clause in normalized for lit in clause})
    started = time.perf_counter()
    model = dpll(normalized)
    elapsed_ms = round((time.perf_counter() - started) * 1000, 3)

    if model is None:
        return {
            "status": "unsat",
            "logic": "propositional-cnf",
            "clauses": normalized,
            "variables": [symbol_name(v, symbols) for v in variables],
            "elapsed_ms": elapsed_ms,
        }

    completed_model = {v: bool(model.get(v, False)) for v in variables}
    return {
        "status": "sat",
        "logic": "propositional-cnf",
        "clauses": normalized,
        "model": {symbol_name(v, symbols): completed_model[v] for v in variables},
        "elapsed_ms": elapsed_ms,
    }


ALLOWED_BINOPS: Dict[type, Callable[[Any, Any], Any]] = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
}
ALLOWED_UNARYOPS: Dict[type, Callable[[Any], Any]] = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
    ast.Not: operator.not_,
}
ALLOWED_CMPOPS: Dict[type, Callable[[Any, Any], bool]] = {
    ast.Eq: operator.eq,
    ast.NotEq: operator.ne,
    ast.Lt: operator.lt,
    ast.LtE: operator.le,
    ast.Gt: operator.gt,
    ast.GtE: operator.ge,
}


def extract_names(expr: str) -> Set[str]:
    tree = ast.parse(expr, mode="eval")
    names: Set[str] = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Name):
            names.add(node.id)
        elif isinstance(node, (ast.Call, ast.Attribute, ast.Subscript, ast.Lambda, ast.Dict, ast.List, ast.Set)):
            raise ValueError(f"Expression interdite dans la contrainte: {expr!r}")
    return names


def safe_eval_ast(node: ast.AST, env: Dict[str, int]) -> Any:
    if isinstance(node, ast.Expression):
        return safe_eval_ast(node.body, env)
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float, bool)):
            return node.value
        raise ValueError("Constante non autorisee")
    if isinstance(node, ast.Name):
        if node.id not in env:
            raise ValueError(f"Variable inconnue: {node.id}")
        return env[node.id]
    if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_BINOPS:
        return ALLOWED_BINOPS[type(node.op)](safe_eval_ast(node.left, env), safe_eval_ast(node.right, env))
    if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_UNARYOPS:
        return ALLOWED_UNARYOPS[type(node.op)](safe_eval_ast(node.operand, env))
    if isinstance(node, ast.BoolOp):
        values = [bool(safe_eval_ast(value, env)) for value in node.values]
        if isinstance(node.op, ast.And):
            return all(values)
        if isinstance(node.op, ast.Or):
            return any(values)
    if isinstance(node, ast.Compare):
        left = safe_eval_ast(node.left, env)
        for op, comparator in zip(node.ops, node.comparators):
            right = safe_eval_ast(comparator, env)
            if type(op) not in ALLOWED_CMPOPS:
                raise ValueError("Comparateur non autorise")
            if not ALLOWED_CMPOPS[type(op)](left, right):
                return False
            left = right
        return True
    raise ValueError(f"Noeud AST non autorise: {type(node).__name__}")


def eval_constraint(expr: str, env: Dict[str, int]) -> bool:
    tree = ast.parse(expr, mode="eval")
    return bool(safe_eval_ast(tree, env))


def normalize_bounds(variables: Sequence[str], bounds: Optional[Dict[str, Sequence[int]]] = None, default_bound: Sequence[int] = (-5, 5)) -> Dict[str, Tuple[int, int]]:
    result: Dict[str, Tuple[int, int]] = {}
    bounds = bounds or {}
    for var in variables:
        raw = bounds.get(var, default_bound)
        if len(raw) != 2:
            raise ValueError(f"Borne invalide pour {var}: {raw}")
        low, high = int(raw[0]), int(raw[1])
        if low > high:
            raise ValueError(f"Borne vide pour {var}: {raw}")
        result[var] = (low, high)
    return result


def solve_smt(constraints: Sequence[str], bounds: Optional[Dict[str, Sequence[int]]] = None, default_bound: Sequence[int] = (-5, 5), max_models: int = 1) -> JSON:
    constraints = [str(c) for c in constraints]
    variables = sorted(set().union(*(extract_names(c) for c in constraints)) if constraints else set())
    normalized_bounds = normalize_bounds(variables, bounds, default_bound)
    domains = [range(normalized_bounds[var][0], normalized_bounds[var][1] + 1) for var in variables]
    started = time.perf_counter()
    checked = 0
    models: List[Dict[str, int]] = []

    for values in itertools.product(*domains):
        env = dict(zip(variables, values))
        checked += 1
        if all(eval_constraint(constraint, env) for constraint in constraints):
            models.append(env)
            if len(models) >= max_models:
                break

    elapsed_ms = round((time.perf_counter() - started) * 1000, 3)
    return {
        "status": "sat" if models else "unsat",
        "logic": "bounded-integer-arithmetic",
        "constraints": constraints,
        "bounds": normalized_bounds,
        "model": models[0] if models else None,
        "models_found": len(models),
        "assignments_checked": checked,
        "elapsed_ms": elapsed_ms,
    }


def sat_smt_tool(arguments: JSON) -> JSON:
    if "clauses" in arguments:
        return solve_sat(arguments["clauses"], symbols=arguments.get("symbols"))
    if "constraints" in arguments:
        return solve_smt(
            arguments["constraints"],
            bounds=arguments.get("bounds"),
            default_bound=arguments.get("default_bound", (-5, 5)),
            max_models=int(arguments.get("max_models", 1)),
        )
    raise ValueError("L'outil SAT/SMT attend soit 'clauses', soit 'constraints'.")

## Outil 2 - Verificateur de preuves

Le verificateur controle des preuves par **resolution propositionnelle**. Les premisses sont des clauses, et chaque etape de preuve doit deriver une nouvelle clause par resolution de deux clauses precedentes.

Notation acceptee : `"P"`, `"~P"`, `"not P"`, ou des clauses comme `['P', '~Q']`.

Une preuve d'insatisfiabilite se termine par la clause vide `[]`.

In [3]:
def normalize_literal(literal: str) -> str:
    text = str(literal).strip()
    text = text.replace("\u00ac", "~")
    if text.lower().startswith("not "):
        text = "~" + text[4:].strip()
    if text.startswith("-"):
        text = "~" + text[1:].strip()
    if text.startswith("~"):
        atom = text[1:].strip()
        if not atom:
            raise ValueError("Litteral negatif vide")
        return "~" + atom
    if not text:
        raise ValueError("Litteral vide")
    return text


def literal_atom(literal: str) -> str:
    literal = normalize_literal(literal)
    return literal[1:] if literal.startswith("~") else literal


def complement(literal: str) -> str:
    literal = normalize_literal(literal)
    return literal[1:] if literal.startswith("~") else "~" + literal


def normalize_literal_clause(raw_clause: Any) -> LiteralClause:
    if raw_clause in (None, ""):
        return frozenset()
    if isinstance(raw_clause, str):
        if raw_clause.strip() in {"[]", "empty", "false", "bot"}:
            return frozenset()
        parts = [part.strip() for part in re.split(r"\s*\|\s*|\s*,\s*", raw_clause) if part.strip()]
    else:
        parts = list(raw_clause)
    literals = frozenset(normalize_literal(part) for part in parts)
    for lit in literals:
        if complement(lit) in literals:
            raise ValueError(f"Clause tautologique interdite dans une preuve: {raw_clause!r}")
    return literals


def format_literal_clause(clause: LiteralClause) -> List[str]:
    return sorted(clause, key=lambda lit: (literal_atom(lit), lit.startswith("~")))


def resolution_resolvents(left: LiteralClause, right: LiteralClause) -> Set[LiteralClause]:
    resolvents: Set[LiteralClause] = set()
    for lit in left:
        comp = complement(lit)
        if comp in right:
            candidate = (left - {lit}) | (right - {comp})
            if all(complement(c) not in candidate for c in candidate):
                resolvents.add(frozenset(candidate))
    return resolvents


def verify_resolution_proof(premises: Sequence[Any], proof: Sequence[JSON], conclusion: Optional[Any] = None) -> JSON:
    premise_clauses = [normalize_literal_clause(clause) for clause in premises]
    derived: Dict[str, LiteralClause] = {f"P{i + 1}": clause for i, clause in enumerate(premise_clauses)}
    trace: List[JSON] = [
        {"id": f"P{i + 1}", "rule": "premise", "clause": format_literal_clause(clause)}
        for i, clause in enumerate(premise_clauses)
    ]

    last_id: Optional[str] = None
    for index, step in enumerate(proof, start=1):
        step_id = str(step.get("id", f"S{index}"))
        rule = str(step.get("rule", "resolve")).lower()
        if step_id in derived:
            return {"valid": False, "error": f"Identifiant deja utilise: {step_id}", "trace": trace}

        if rule == "premise":
            clause = normalize_literal_clause(step.get("clause", []))
            if clause not in premise_clauses:
                return {"valid": False, "error": f"La clause {format_literal_clause(clause)} n'est pas une premisse", "trace": trace}
        elif rule == "copy":
            source = str(step.get("from"))
            if source not in derived:
                return {"valid": False, "error": f"Source inconnue: {source}", "trace": trace}
            clause = derived[source]
        elif rule == "resolve":
            sources = step.get("from", [])
            if not isinstance(sources, list) or len(sources) != 2:
                return {"valid": False, "error": "Une resolution attend exactement deux sources", "trace": trace}
            left_id, right_id = map(str, sources)
            if left_id not in derived or right_id not in derived:
                return {"valid": False, "error": f"Sources inconnues: {sources}", "trace": trace}
            possible = resolution_resolvents(derived[left_id], derived[right_id])
            clause = normalize_literal_clause(step.get("clause", []))
            if clause not in possible:
                return {
                    "valid": False,
                    "error": "Clause resolvante incorrecte",
                    "expected_one_of": [format_literal_clause(c) for c in sorted(possible, key=lambda c: format_literal_clause(c))],
                    "got": format_literal_clause(clause),
                    "trace": trace,
                }
        else:
            return {"valid": False, "error": f"Regle inconnue: {rule}", "trace": trace}

        derived[step_id] = clause
        last_id = step_id
        trace.append({"id": step_id, "rule": rule, "clause": format_literal_clause(clause), "from": step.get("from")})

    final_clause = derived[last_id] if last_id else (premise_clauses[-1] if premise_clauses else frozenset())
    target_clause = normalize_literal_clause(conclusion) if conclusion is not None else final_clause
    valid = final_clause == target_clause
    return {
        "valid": valid,
        "conclusion": format_literal_clause(final_clause),
        "target": format_literal_clause(target_clause),
        "is_refutation": len(final_clause) == 0,
        "steps_checked": len(proof),
        "trace": trace,
        "error": None if valid else "La conclusion finale ne correspond pas a la cible",
    }


def proof_tool(arguments: JSON) -> JSON:
    return verify_resolution_proof(
        premises=arguments.get("premises", []),
        proof=arguments.get("proof", []),
        conclusion=arguments.get("conclusion"),
    )

## Outil 3 - Raisonneur OWL-lite

Ce raisonneur travaille sur des triplets `(sujet, predicat, objet)` et applique une saturation simple :

- transitivite de `subClassOf` et `subPropertyOf` ;
- propagation des types par les sous-classes ;
- propagation des assertions par les sous-proprietes ;
- `equivalentClass`, `inverseOf`, `domain`, `range`, `transitiveProperty` ;
- detection d'incoherence pour `disjointWith`.

C'est un fragment OWL/RDFS volontairement limite, suffisant pour demontrer l'encapsulation d'un raisonneur symbolique.

In [4]:
RESERVED_PREDICATES = {
    "type",
    "subClassOf",
    "equivalentClass",
    "disjointWith",
    "subPropertyOf",
    "domain",
    "range",
    "inverseOf",
    "transitiveProperty",
}


def normalize_triple(raw: Any) -> Triple:
    if isinstance(raw, dict):
        return (str(raw["subject"]), str(raw["predicate"]), str(raw["object"]))
    if len(raw) != 3:
        raise ValueError(f"Triplet invalide: {raw!r}")
    return (str(raw[0]), str(raw[1]), str(raw[2]))


def is_property_assertion(triple: Triple) -> bool:
    _, predicate, _ = triple
    return predicate not in RESERVED_PREDICATES


def owl_reasoner(arguments: JSON) -> JSON:
    base_triples: Set[Triple] = {normalize_triple(t) for t in arguments.get("triples", [])}
    triples: Set[Triple] = set(base_triples)
    max_iterations = int(arguments.get("max_iterations", 50))
    iterations = 0

    def add(triple: Triple) -> bool:
        if triple not in triples:
            triples.add(triple)
            return True
        return False

    changed = True
    while changed and iterations < max_iterations:
        changed = False
        iterations += 1
        snapshot = set(triples)

        subclasses = {(s, o) for s, p, o in snapshot if p == "subClassOf"}
        subproperties = {(s, o) for s, p, o in snapshot if p == "subPropertyOf"}
        property_assertions = {t for t in snapshot if is_property_assertion(t)}

        for a, p, b in snapshot:
            if p == "equivalentClass":
                changed |= add((a, "subClassOf", b))
                changed |= add((b, "subClassOf", a))
            if p == "inverseOf":
                changed |= add((b, "inverseOf", a))

        subclasses = {(s, o) for s, p, o in triples if p == "subClassOf"}
        for a, b in list(subclasses):
            for c, d in list(subclasses):
                if b == c:
                    changed |= add((a, "subClassOf", d))

        subproperties = {(s, o) for s, p, o in triples if p == "subPropertyOf"}
        for a, b in list(subproperties):
            for c, d in list(subproperties):
                if b == c:
                    changed |= add((a, "subPropertyOf", d))

        for individual, predicate, cls in list(triples):
            if predicate == "type":
                for child, parent in subclasses:
                    if cls == child:
                        changed |= add((individual, "type", parent))

        for subject, predicate, obj in list(property_assertions):
            for child, parent in subproperties:
                if predicate == child:
                    changed |= add((subject, parent, obj))

        domains = {(prop, cls) for prop, p, cls in triples if p == "domain"}
        ranges = {(prop, cls) for prop, p, cls in triples if p == "range"}
        inverse_pairs = {(prop, inv) for prop, p, inv in triples if p == "inverseOf"}
        transitive_props = {prop for prop, p, value in triples if p == "transitiveProperty" and value.lower() in {"true", "1", "yes"}}
        property_assertions = {t for t in triples if is_property_assertion(t)}

        for subject, predicate, obj in list(property_assertions):
            for prop, cls in domains:
                if predicate == prop:
                    changed |= add((subject, "type", cls))
            for prop, cls in ranges:
                if predicate == prop:
                    changed |= add((obj, "type", cls))
            for prop, inverse in inverse_pairs:
                if predicate == prop:
                    changed |= add((obj, inverse, subject))

        for prop in transitive_props:
            prop_edges = [(s, o) for s, p, o in triples if p == prop]
            for a, b in prop_edges:
                for c, d in prop_edges:
                    if b == c:
                        changed |= add((a, prop, d))

    disjoint_pairs = {(a, b) for a, p, b in triples if p == "disjointWith"}
    disjoint_pairs |= {(b, a) for a, b in disjoint_pairs}
    types_by_individual: Dict[str, Set[str]] = {}
    for individual, predicate, cls in triples:
        if predicate == "type":
            types_by_individual.setdefault(individual, set()).add(cls)

    inconsistencies: List[JSON] = []
    for individual, classes in sorted(types_by_individual.items()):
        for cls_a, cls_b in itertools.combinations(sorted(classes), 2):
            if (cls_a, cls_b) in disjoint_pairs:
                inconsistencies.append({
                    "individual": individual,
                    "classes": [cls_a, cls_b],
                    "reason": "disjointWith",
                })

    entailed = sorted(triples)
    inferred = sorted(triples - base_triples)
    return {
        "consistent": len(inconsistencies) == 0,
        "iterations": iterations,
        "input_triples": sorted(base_triples),
        "inferred_triples": inferred,
        "entailed_triples": entailed,
        "inconsistencies": inconsistencies,
        "limit_reached": iterations >= max_iterations,
    }

## Adaptateur LLM et serveur JSON-RPC/MCP

Cette section encapsule les outils derriere une interface type MCP. La methode principale est `handle(request)`, qui prend une requete JSON-RPC et renvoie une reponse JSON-RPC.

Les resultats sont fournis en deux formes :

- `structuredContent` : donnees directement exploitables par programme ;
- `content` : resume textuel utilisable par un LLM.

In [5]:
TOOL_SCHEMAS: List[JSON] = [
    {
        "name": "symbolic.sat_smt.solve",
        "description": "Resout une formule SAT en CNF ou un petit probleme SMT borne sur entiers.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "clauses": {"type": "array", "description": "CNF sous forme de clauses d'entiers."},
                "symbols": {"type": ["array", "object"], "description": "Noms optionnels des variables SAT."},
                "constraints": {"type": "array", "items": {"type": "string"}},
                "bounds": {"type": "object", "description": "Bornes par variable, ex: {'x': [0, 5]}."},
                "default_bound": {"type": "array", "items": {"type": "integer"}},
                "max_models": {"type": "integer", "default": 1},
            },
        },
    },
    {
        "name": "symbolic.proof.verify",
        "description": "Verifie une preuve par resolution propositionnelle.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "premises": {"type": "array", "description": "Clauses de depart."},
                "proof": {"type": "array", "description": "Etapes de resolution."},
                "conclusion": {"description": "Clause cible optionnelle."},
            },
            "required": ["premises", "proof"],
        },
    },
    {
        "name": "symbolic.owl.reason",
        "description": "Raisonnement OWL-lite/RDFS par saturation de triplets.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "triples": {"type": "array", "description": "Triplets (sujet, predicat, objet)."},
                "max_iterations": {"type": "integer", "default": 50},
            },
            "required": ["triples"],
        },
    },
]

TOOL_FUNCTIONS: Dict[str, Callable[[JSON], JSON]] = {
    "symbolic.sat_smt.solve": sat_smt_tool,
    "symbolic.proof.verify": proof_tool,
    "symbolic.owl.reason": owl_reasoner,
}


@dataclass
class Session:
    id: str
    created_at: float = field(default_factory=time.time)
    events: List[JSON] = field(default_factory=list)
    memory: Dict[str, Any] = field(default_factory=dict)

    def remember(self, event: JSON) -> str:
        event_id = f"evt_{len(self.events) + 1}"
        record = {"id": event_id, "at": round(time.time(), 3), **event}
        self.events.append(record)
        self.memory[event_id] = record
        return event_id


class SymbolicMCPServer:
    def __init__(self) -> None:
        self.sessions: Dict[str, Session] = {}
        self.tools = TOOL_FUNCTIONS
        self.tool_schemas = TOOL_SCHEMAS

    def _ok(self, request_id: Any, result: JSON) -> JSON:
        return {"jsonrpc": "2.0", "id": request_id, "result": result}

    def _error(self, request_id: Any, code: int, message: str, data: Optional[Any] = None) -> JSON:
        error: JSON = {"code": code, "message": message}
        if data is not None:
            error["data"] = data
        return {"jsonrpc": "2.0", "id": request_id, "error": error}

    def create_session(self) -> Session:
        session = Session(id="sess_" + uuid.uuid4().hex[:12])
        self.sessions[session.id] = session
        return session

    def get_session(self, session_id: Optional[str]) -> Session:
        if session_id and session_id in self.sessions:
            return self.sessions[session_id]
        return self.create_session()

    def summarize_tool_result(self, name: str, result: JSON) -> str:
        if name == "symbolic.sat_smt.solve":
            if result["status"] == "sat":
                return f"Instance satisfiable. Modele: {result.get('model')}"
            return "Instance insatisfiable. Aucun modele trouve dans la logique demandee."
        if name == "symbolic.proof.verify":
            return "Preuve valide." if result.get("valid") else f"Preuve invalide: {result.get('error')}"
        if name == "symbolic.owl.reason":
            status = "coherente" if result.get("consistent") else "incoherente"
            return f"Ontologie {status}. {len(result.get('inferred_triples', []))} triplets infere(s)."
        return json.dumps(result, ensure_ascii=False)

    def handle(self, request: JSON) -> JSON:
        request_id = request.get("id")
        try:
            if request.get("jsonrpc") != "2.0":
                return self._error(request_id, -32600, "Requete JSON-RPC invalide: champ jsonrpc attendu a '2.0'.")

            method = request.get("method")
            params = request.get("params", {}) or {}

            if method == "initialize":
                return self._ok(request_id, {
                    "protocolVersion": "2024-11-05",
                    "serverInfo": {"name": "j3-symbolic-mcp-notebook", "version": "1.0.0"},
                    "capabilities": {"tools": {"listChanged": False}},
                })

            if method == "ping":
                return self._ok(request_id, {"ok": True})

            if method == "sessions/create":
                session = self.create_session()
                return self._ok(request_id, {"session_id": session.id, "created_at": session.created_at})

            if method == "sessions/get":
                session_id = params.get("session_id")
                if session_id not in self.sessions:
                    return self._error(request_id, -32004, f"Session inconnue: {session_id}")
                session = self.sessions[session_id]
                return self._ok(request_id, {"session_id": session.id, "events": session.events})

            if method == "tools/list":
                return self._ok(request_id, {"tools": self.tool_schemas})

            if method == "tools/call":
                name = params.get("name")
                arguments = dict(params.get("arguments", {}) or {})
                session = self.get_session(params.get("session_id") or arguments.pop("session_id", None))
                if name not in self.tools:
                    return self._error(request_id, -32602, f"Outil inconnu: {name}")

                result = self.tools[name](arguments)
                event_id = session.remember({"tool": name, "arguments": arguments, "result": result})
                summary = self.summarize_tool_result(name, result)
                return self._ok(request_id, {
                    "content": [{"type": "text", "text": summary}],
                    "structuredContent": result,
                    "session_id": session.id,
                    "event_id": event_id,
                })

            return self._error(request_id, -32601, f"Methode inconnue: {method}")
        except Exception as exc:
            return self._error(request_id, -32603, "Erreur interne pendant le traitement", {"exception": type(exc).__name__, "message": str(exc)})


def serve_stdio(server: SymbolicMCPServer) -> None:
    """Boucle stdio minimale: une requete JSON-RPC par ligne, une reponse par ligne."""
    for line in sys.stdin:
        if not line.strip():
            continue
        response = server.handle(json.loads(line))
        print(json.dumps(response, ensure_ascii=False), flush=True)

## Demonstration de l'API

Les cellules suivantes simulent les appels qu'un client MCP ou un LLM outille enverrait au serveur.

In [6]:
server = SymbolicMCPServer()

pretty(server.handle({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {},
}))

session_response = server.handle({
    "jsonrpc": "2.0",
    "id": 2,
    "method": "sessions/create",
    "params": {},
})
session_id = session_response["result"]["session_id"]
print("Session:", session_id)

{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "protocolVersion": "2024-11-05",
    "serverInfo": {
      "name": "j3-symbolic-mcp-notebook",
      "version": "1.0.0"
    },
    "capabilities": {
      "tools": {
        "listChanged": false
      }
    }
  }
}
Session: sess_5b2f2ac17b51


In [7]:
pretty(server.handle({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/list",
    "params": {},
}))

{
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "tools": [
      {
        "name": "symbolic.sat_smt.solve",
        "description": "Resout une formule SAT en CNF ou un petit probleme SMT borne sur entiers.",
        "inputSchema": {
          "type": "object",
          "properties": {
            "clauses": {
              "type": "array",
              "description": "CNF sous forme de clauses d'entiers."
            },
            "symbols": {
              "type": [
                "array",
                "object"
              ],
              "description": "Noms optionnels des variables SAT."
            },
            "constraints": {
              "type": "array",
              "items": {
                "type": "string"
              }
            },
            "bounds": {
              "type": "object",
              "description": "Bornes par variable, ex: {'x': [0, 5]}."
            },
            "default_bound": {
              "type": "array",
              "items

### Exemple SAT

Formule : `(P ou Q) et (non P ou Q) et (P ou non Q)`.

In [8]:
sat_response = server.handle({
    "jsonrpc": "2.0",
    "id": 4,
    "method": "tools/call",
    "params": {
        "session_id": session_id,
        "name": "symbolic.sat_smt.solve",
        "arguments": {
            "clauses": [[1, 2], [-1, 2], [1, -2]],
            "symbols": ["P", "Q"],
        },
    },
})
pretty(sat_response)

{
  "jsonrpc": "2.0",
  "id": 4,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Instance satisfiable. Modele: {'P': True, 'Q': True}"
      }
    ],
    "structuredContent": {
      "status": "sat",
      "logic": "propositional-cnf",
      "clauses": [
        [
          1,
          2
        ],
        [
          -1,
          2
        ],
        [
          1,
          -2
        ]
      ],
      "model": {
        "P": true,
        "Q": true
      },
      "elapsed_ms": 0.018
    },
    "session_id": "sess_5b2f2ac17b51",
    "event_id": "evt_1"
  }
}


### Exemple SMT borne

On cherche des entiers `x` et `y` tels que `x + y == 7`, `x >= 2`, `y >= 2`, avec des bornes finies.

In [9]:
smt_response = server.handle({
    "jsonrpc": "2.0",
    "id": 5,
    "method": "tools/call",
    "params": {
        "session_id": session_id,
        "name": "symbolic.sat_smt.solve",
        "arguments": {
            "constraints": ["x + y == 7", "x >= 2", "y >= 2", "x < y"],
            "bounds": {"x": [0, 10], "y": [0, 10]},
        },
    },
})
pretty(smt_response)

{
  "jsonrpc": "2.0",
  "id": 5,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Instance satisfiable. Modele: {'x': 2, 'y': 5}"
      }
    ],
    "structuredContent": {
      "status": "sat",
      "logic": "bounded-integer-arithmetic",
      "constraints": [
        "x + y == 7",
        "x >= 2",
        "y >= 2",
        "x < y"
      ],
      "bounds": {
        "x": [
          0,
          10
        ],
        "y": [
          0,
          10
        ]
      },
      "model": {
        "x": 2,
        "y": 5
      },
      "models_found": 1,
      "assignments_checked": 28,
      "elapsed_ms": 0.299
    },
    "session_id": "sess_5b2f2ac17b51",
    "event_id": "evt_2"
  }
}


### Exemple de verification de preuve

Premisses : `(P ou Q)`, `non P`, `non Q`.

La resolution derive d'abord `Q`, puis la clause vide. La clause vide prouve l'insatisfiabilite de l'ensemble de premisses.

In [10]:
proof_response = server.handle({
    "jsonrpc": "2.0",
    "id": 6,
    "method": "tools/call",
    "params": {
        "session_id": session_id,
        "name": "symbolic.proof.verify",
        "arguments": {
            "premises": [["P", "Q"], ["~P"], ["~Q"]],
            "proof": [
                {"id": "S1", "rule": "resolve", "from": ["P1", "P2"], "clause": ["Q"]},
                {"id": "S2", "rule": "resolve", "from": ["S1", "P3"], "clause": []},
            ],
            "conclusion": [],
        },
    },
})
pretty(proof_response)

{
  "jsonrpc": "2.0",
  "id": 6,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Preuve valide."
      }
    ],
    "structuredContent": {
      "valid": true,
      "conclusion": [],
      "target": [],
      "is_refutation": true,
      "steps_checked": 2,
      "trace": [
        {
          "id": "P1",
          "rule": "premise",
          "clause": [
            "P",
            "Q"
          ]
        },
        {
          "id": "P2",
          "rule": "premise",
          "clause": [
            "~P"
          ]
        },
        {
          "id": "P3",
          "rule": "premise",
          "clause": [
            "~Q"
          ]
        },
        {
          "id": "S1",
          "rule": "resolve",
          "clause": [
            "Q"
          ],
          "from": [
            "P1",
            "P2"
          ]
        },
        {
          "id": "S2",
          "rule": "resolve",
          "clause": [],
          "from": [
            

### Exemple OWL-lite

On decrit une petite ontologie :

- `Dog subClassOf Mammal`
- `Mammal subClassOf Animal`
- `fido type Dog`
- `eats domain Animal`
- `eats range Food`
- `parentOf inverseOf childOf`
- `ancestorOf transitiveProperty true`

Le raisonneur doit inferer par exemple que `fido type Animal`.

In [11]:
owl_response = server.handle({
    "jsonrpc": "2.0",
    "id": 7,
    "method": "tools/call",
    "params": {
        "session_id": session_id,
        "name": "symbolic.owl.reason",
        "arguments": {
            "triples": [
                ["Dog", "subClassOf", "Mammal"],
                ["Mammal", "subClassOf", "Animal"],
                ["fido", "type", "Dog"],
                ["eats", "domain", "Animal"],
                ["eats", "range", "Food"],
                ["fido", "eats", "kibble"],
                ["parentOf", "inverseOf", "childOf"],
                ["alice", "parentOf", "bob"],
                ["ancestorOf", "transitiveProperty", "true"],
                ["alice", "ancestorOf", "bob"],
                ["bob", "ancestorOf", "carol"],
            ]
        },
    },
})
pretty(owl_response)

{
  "jsonrpc": "2.0",
  "id": 7,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Ontologie coherente. 7 triplets infere(s)."
      }
    ],
    "structuredContent": {
      "consistent": true,
      "iterations": 2,
      "input_triples": [
        [
          "Dog",
          "subClassOf",
          "Mammal"
        ],
        [
          "Mammal",
          "subClassOf",
          "Animal"
        ],
        [
          "alice",
          "ancestorOf",
          "bob"
        ],
        [
          "alice",
          "parentOf",
          "bob"
        ],
        [
          "ancestorOf",
          "transitiveProperty",
          "true"
        ],
        [
          "bob",
          "ancestorOf",
          "carol"
        ],
        [
          "eats",
          "domain",
          "Animal"
        ],
        [
          "eats",
          "range",
          "Food"
        ],
        [
          "fido",
          "eats",
          "kibble"
        ],
 

### Inspection du contexte de session

Chaque appel d'outil est conserve dans la session. Cela permet a un orchestrateur LLM de reutiliser les resultats precedents, de tracer les decisions et de construire un dialogue multi-etapes.

In [12]:
pretty(server.handle({
    "jsonrpc": "2.0",
    "id": 8,
    "method": "sessions/get",
    "params": {"session_id": session_id},
}))

{
  "jsonrpc": "2.0",
  "id": 8,
  "result": {
    "session_id": "sess_5b2f2ac17b51",
    "events": [
      {
        "id": "evt_1",
        "at": 1784197650.11,
        "tool": "symbolic.sat_smt.solve",
        "arguments": {
          "clauses": [
            [
              1,
              2
            ],
            [
              -1,
              2
            ],
            [
              1,
              -2
            ]
          ],
          "symbols": [
            "P",
            "Q"
          ]
        },
        "result": {
          "status": "sat",
          "logic": "propositional-cnf",
          "clauses": [
            [
              1,
              2
            ],
            [
              -1,
              2
            ],
            [
              1,
              -2
            ]
          ],
          "model": {
            "P": true,
            "Q": true
          },
          "elapsed_ms": 0.018
        }
      },
      {
        "id": "evt_2",
  

## Traduction bidirectionnelle LLM <-> outils

Un serveur MCP sert generalement d'interface entre un LLM et des outils. La traduction peut etre tres simple : un message utilisateur est transforme en appel d'outil, puis le resultat symbolique est transforme en reponse textuelle.

La fonction suivante est volontairement minimale, mais montre le principe d'orchestration.

In [13]:
def llm_to_tool_call(message: str) -> JSON:
    text = message.lower()
    if "preuve" in text or "proof" in text:
        return {"name": "symbolic.proof.verify", "arguments": {}}
    if "owl" in text or "ontologie" in text or "classe" in text:
        return {"name": "symbolic.owl.reason", "arguments": {"triples": []}}
    if "contrainte" in text or "smt" in text:
        return {"name": "symbolic.sat_smt.solve", "arguments": {"constraints": []}}
    return {"name": "symbolic.sat_smt.solve", "arguments": {"clauses": []}}


def tool_result_to_llm_answer(response: JSON) -> str:
    if "error" in response:
        return "L'appel d'outil a echoue: " + response["error"]["message"]
    content = response["result"].get("content", [])
    if content:
        return content[0].get("text", "")
    return json.dumps(response["result"].get("structuredContent", {}), ensure_ascii=False)

example_intent = llm_to_tool_call("Peux-tu verifier cette ontologie OWL ?")
pretty(example_intent)
print(tool_result_to_llm_answer(owl_response))

{
  "name": "symbolic.owl.reason",
  "arguments": {
    "triples": []
  }
}
Ontologie coherente. 7 triplets infere(s).


## Evaluation

L'evaluation ci-dessous mesure :

- la precision : le resultat attendu est-il obtenu sur des cas de reference ?
- la robustesse : le serveur renvoie-t-il une erreur JSON-RPC propre sur des entrees invalides ?

In [14]:
def call_tool(server: SymbolicMCPServer, tool_name: str, arguments: JSON) -> JSON:
    response = server.handle({
        "jsonrpc": "2.0",
        "id": "bench",
        "method": "tools/call",
        "params": {"name": tool_name, "arguments": arguments},
    })
    if "error" in response:
        return {"_error": response["error"]}
    return response["result"]["structuredContent"]


benchmarks: List[JSON] = [
    {
        "name": "sat_cnf_sat",
        "tool": "symbolic.sat_smt.solve",
        "arguments": {"clauses": [[1, 2], [-1, 2], [1, -2]], "symbols": ["P", "Q"]},
        "check": lambda r: r.get("status") == "sat" and r.get("model", {}).get("P") is True and r.get("model", {}).get("Q") is True,
    },
    {
        "name": "sat_cnf_unsat",
        "tool": "symbolic.sat_smt.solve",
        "arguments": {"clauses": [[1], [-1]], "symbols": ["P"]},
        "check": lambda r: r.get("status") == "unsat",
    },
    {
        "name": "smt_sat",
        "tool": "symbolic.sat_smt.solve",
        "arguments": {"constraints": ["x + y == 3", "x >= 1", "y >= 1"], "bounds": {"x": [0, 3], "y": [0, 3]}},
        "check": lambda r: r.get("status") == "sat" and r.get("model", {}).get("x", 0) + r.get("model", {}).get("y", 0) == 3,
    },
    {
        "name": "smt_unsat",
        "tool": "symbolic.sat_smt.solve",
        "arguments": {"constraints": ["x >= 3", "x <= 1"], "bounds": {"x": [0, 5]}},
        "check": lambda r: r.get("status") == "unsat",
    },
    {
        "name": "proof_valid_refutation",
        "tool": "symbolic.proof.verify",
        "arguments": {
            "premises": [["P", "Q"], ["~P"], ["~Q"]],
            "proof": [
                {"id": "S1", "rule": "resolve", "from": ["P1", "P2"], "clause": ["Q"]},
                {"id": "S2", "rule": "resolve", "from": ["S1", "P3"], "clause": []},
            ],
            "conclusion": [],
        },
        "check": lambda r: r.get("valid") is True and r.get("is_refutation") is True,
    },
    {
        "name": "proof_invalid_step",
        "tool": "symbolic.proof.verify",
        "arguments": {
            "premises": [["P", "Q"], ["~P"]],
            "proof": [{"id": "S1", "rule": "resolve", "from": ["P1", "P2"], "clause": ["P"]}],
        },
        "check": lambda r: r.get("valid") is False,
    },
    {
        "name": "owl_type_propagation",
        "tool": "symbolic.owl.reason",
        "arguments": {"triples": [["Dog", "subClassOf", "Animal"], ["fido", "type", "Dog"]]},
        "check": lambda r: ["fido", "type", "Animal"] in [list(t) for t in r.get("entailed_triples", [])],
    },
    {
        "name": "owl_disjoint_inconsistency",
        "tool": "symbolic.owl.reason",
        "arguments": {"triples": [["Dog", "disjointWith", "Cat"], ["milo", "type", "Dog"], ["milo", "type", "Cat"]]},
        "check": lambda r: r.get("consistent") is False and len(r.get("inconsistencies", [])) == 1,
    },
]

rows: List[JSON] = []
for benchmark in benchmarks:
    result = call_tool(server, benchmark["tool"], benchmark["arguments"])
    passed = bool(benchmark["check"](result))
    if "status" in result:
        summary = result["status"]
    elif "valid" in result:
        summary = result["valid"]
    elif "consistent" in result:
        summary = result["consistent"]
    else:
        summary = None
    rows.append({
        "benchmark": benchmark["name"],
        "tool": benchmark["tool"],
        "passed": passed,
        "summary": summary,
    })

accuracy = sum(row["passed"] for row in rows) / len(rows)

malformed_requests = [
    {"jsonrpc": "2.0", "id": "bad1", "method": "tools/call", "params": {"name": "unknown", "arguments": {}}},
    {"jsonrpc": "2.0", "id": "bad2", "method": "tools/call", "params": {"name": "symbolic.sat_smt.solve", "arguments": {"constraints": ["__import__('os').system('echo nope')"]}}},
    {"jsonrpc": "1.0", "id": "bad3", "method": "ping", "params": {}},
]
robustness_responses = [server.handle(request) for request in malformed_requests]
robustness = all("error" in response for response in robustness_responses)

print(f"Precision benchmark: {accuracy:.0%} ({sum(row['passed'] for row in rows)}/{len(rows)})")
print(f"Robustesse erreurs JSON-RPC: {'OK' if robustness else 'ECHEC'}")

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    pretty(rows)

Precision benchmark: 100% (8/8)
Robustesse erreurs JSON-RPC: OK
[
  {
    "benchmark": "sat_cnf_sat",
    "tool": "symbolic.sat_smt.solve",
    "passed": true,
    "summary": "sat"
  },
  {
    "benchmark": "sat_cnf_unsat",
    "tool": "symbolic.sat_smt.solve",
    "passed": true,
    "summary": "unsat"
  },
  {
    "benchmark": "smt_sat",
    "tool": "symbolic.sat_smt.solve",
    "passed": true,
    "summary": "sat"
  },
  {
    "benchmark": "smt_unsat",
    "tool": "symbolic.sat_smt.solve",
    "passed": true,
    "summary": "unsat"
  },
  {
    "benchmark": "proof_valid_refutation",
    "tool": "symbolic.proof.verify",
    "passed": true,
    "summary": true
  },
  {
    "benchmark": "proof_invalid_step",
    "tool": "symbolic.proof.verify",
    "passed": true,
    "summary": false
  },
  {
    "benchmark": "owl_type_propagation",
    "tool": "symbolic.owl.reason",
    "passed": true,
    "summary": true
  },
  {
    "benchmark": "owl_disjoint_inconsistency",
    "tool": "symbolic.o

## Conclusion

Le notebook couvre les objectifs du projet :

- implementation d'un serveur JSON-RPC inspire MCP ;
- encapsulation de trois outils symboliques ;
- gestion des sessions, contexte et traduction resultat outil -> resume LLM ;
- documentation par schemas d'outils et exemples ;
- evaluation sur benchmarks de raisonnement.

Limites principales : le mini-SMT est borne et entier, le raisonneur OWL est un fragment OWL-lite, et la boucle stdio n'est pas lancee automatiquement dans le notebook. Ces limites sont assumees pour garder un livrable pedagogique, reproductible et executable sans installation externe.